# 03 · Auditoría de Go y figuras del informe

**Laboratorio en Python, no sistema.** Patrón del TP: dos implementaciones; si discrepan, hay un bug
en una de las dos. Este notebook:

1. **Recalcula silver y gold desde el bronze** con `aire_lab` y la receta de `configs/kmeans.json`,
   y los compara con los de Go: conteos por regla iguales, silver celda por celda, gold con
   tolerancia 1e-9 después del z-score; cualquier diferencia se lista fila a fila.
2. **Audita el K-Means de Go**: asignaciones (centroide más cercano, empate al menor índice),
   tamaños, inercia, que los centroides sean un punto fijo de Lloyd, y la comparación con
   scikit-learn con correspondencia de etiquetas (asignación húngara) y tolerancia.
3. **Audita `regimenes.json`**: perfil, etiqueta ECA y distribución por hora y estación.
4. **Dibuja las figuras del informe** (PNG, 150 dpi, en español) sobre las salidas de Go: perfiles
   por régimen, régimen × hora, régimen × estación, **un episodio real con los regímenes
   coloreados**, cobertura por departamento y la curva del codo.

Cada comprobación queda como OK, DIFERENCIA u OMITIDA (si falta la salida de Go); el notebook no
se cae porque Go no haya corrido todavía: las figuras usan entonces los regímenes de scikit-learn
y lo dicen.

| Entra | Sale |
|---|---|
| `data/bronze/*.csv`, `configs/kmeans.json`, `configs/limpieza.json` | `reports/auditoria.md`, `reports/auditoria.json` |
| Go: `data/silver/oefa_ancho.csv`, `data/gold/oefa_features.csv`, `data/gold/asignaciones.csv` | figuras en `docs/informe/generado/figuras/` (local) o `reports/figuras/` en Drive (Colab) |
| Go: `reports/limpieza.json`, `estandarizacion.json`, `kmeans.json`, `regimenes.json` | |

## 0. Entorno

In [ ]:
# Colab trae casi todo; esta celda instala solo lo que falte (fuera de Colab no hace nada si ya está).
import importlib.util, subprocess, sys
PAQUETES = {"scikit-learn": "sklearn", "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib"}
faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

In [ ]:
# ── Entorno: Drive, raíz de las capas y código compartido (notebooks/aire_lab.py) ──────────────
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:  # fuera de Colab no se monta nada y la celda no falla
    from google.colab import drive
    drive.mount("/content/drive")

# Parámetros de ubicación: lo único que hay que tocar.
RAIZ_DRIVE = Path("/content/drive/MyDrive/aire-distribuido")  # capas medallón en Drive
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")   # alternativa local (p. ej. el sintético); None = repo
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
RAMAS = ["develop", "feature/notebooks"]   # de dónde clonar aire_lab.py en Colab, en ese orden
ACTUALIZAR_CODIGO = False                  # True: vuelve a clonar (si cambió aire_lab.py)


def ubicar_repo() -> Path:
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "notebooks" / "aire_lab.py").exists():
            return c
    destino = Path("/content/aire-distribuido")
    if ACTUALIZAR_CODIGO:
        subprocess.run(["rm", "-rf", str(destino)], check=False)
    for rama in RAMAS:
        if (destino / "notebooks" / "aire_lab.py").exists():
            break
        subprocess.run(["rm", "-rf", str(destino)], check=False)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino)],
                       check=False)
    if not (destino / "notebooks" / "aire_lab.py").exists():
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: clona el repo o súbelo junto al notebook.")
    return destino


REPO = ubicar_repo()
sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (RAIZ_DRIVE if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
SINTETICO = (RAIZ / "SINTETICO.txt").exists()
REPORTS.mkdir(parents=True, exist_ok=True)

import importlib
import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
print(f"Colab: {EN_COLAB}\nrepo: {REPO}\nraíz de las capas: {RAIZ}")
if SINTETICO:
    print("AVISO: la raíz tiene SINTETICO.txt. Son datos sintéticos de prueba, no mediciones de OEFA.")

In [ ]:
import json, math, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)

# Tolerancias de la auditoría.
TOL_SILVER = 1e-12      # relativa: los dos parsean el mismo texto, deberían ser idénticos
TOL_GOLD = 1e-9         # absoluta, después del z-score (spec de calibración §4)
TOL_ESTAND = 1e-9       # relativa, medias y desviaciones
TOL_PUNTO_FIJO = 1e-6   # centroide de Go frente a la media de sus filas asignadas
TOL_SKLEARN = 1e-3      # centroides de Go frente a los de scikit-learn, tras el emparejamiento

# Figuras: en Colab a Drive (reports/figuras); en local al informe. AIRE_FIGURAS pisa a los dos.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
FIGURAS.mkdir(parents=True, exist_ok=True)
print("Figuras en", FIGURAS)

GO = {
    "silver": DATA / "silver" / "oefa_ancho.csv",
    "gold": DATA / "gold" / "oefa_features.csv",
    "asignaciones": DATA / "gold" / "asignaciones.csv",
    "limpieza": REPORTS / "limpieza.json",
    "estandarizacion": REPORTS / "estandarizacion.json",
    "kmeans": REPORTS / "kmeans.json",
    "regimenes": REPORTS / "regimenes.json",
}
display(pd.DataFrame({"ruta": {k: str(v) for k, v in GO.items()}, "existe": {k: v.exists() for k, v in GO.items()}}))

CHEQUEOS = []


def chequeo(nombre, ok, detalle, omitido=False, aviso=False):
    estado = "OMITIDO" if omitido else ("OK" if ok else ("AVISO" if aviso else "DIFERENCIA"))
    CHEQUEOS.append({"chequeo": nombre, "estado": estado, "detalle": detalle})
    print(f"[{estado}] {nombre}: {detalle}")


def leer_json(ruta):
    with open(ruta, encoding="utf-8") as fh:
        return json.load(fh)


def primera(d, claves, defecto=None):
    for c in claves:
        if isinstance(d, dict) and c in d and d[c] is not None:
            return d[c]
    return defecto


def guardar(fig, nombre):
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIGURAS / nombre, dpi=150)
    plt.show()

## 1. Receta

In [ ]:
try:
    receta, origen_receta = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
except FileNotFoundError:
    receta, origen_receta = leer_json(REPORTS / "kmeans_propuesto.json"), str(REPORTS / "kmeans_propuesto.json")
    print("AVISO: no hay configs/kmeans.json; se usa la propuesta del notebook 02.")
FEATS, LOGS, NIVEL, K = receta["features"], receta.get("log1p", []), receta.get("nivel", "absoluto"), int(receta["k"])
print(f"Receta ({origen_receta}):", json.dumps(receta))

## 2. Silver: bronze → silver en Python frente al de Go

Las mismas reglas que el notebook 01 (`aire_lab.limpiar`, en el orden documentado ahí).

In [ ]:
cfg, origen_cfg = al.cargar_config([c / "limpieza.json" for c in CONFIGS])
print("Configuración de limpieza:", origen_cfg)
bronces = sorted((DATA / "bronze").glob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
largo = ancho_py = rep_py = None
if bronces:
    t0 = time.perf_counter()
    largo, info_lectura = al.leer_bronze(bronces[0], cfg)
    validos, rep_py = al.limpiar(largo, cfg)
    ancho_py = al.pivotar(validos)
    print(f"{bronces[0].name}: {len(largo):,} filas largas → {len(ancho_py):,} filas anchas en {time.perf_counter() - t0:.1f} s")
else:
    print("AVISO: no hay bronze; no se puede recalcular el silver.")

silver_go = al.leer_csv_contrato(GO["silver"]) if GO["silver"].exists() else None

# Conteos por regla.
if rep_py is not None and GO["limpieza"].exists():
    lim_go = leer_json(GO["limpieza"])
    reglas_go = primera(lim_go, ["reglas", "conteos", "descartes"], {})
    claves = list(dict.fromkeys(list(rep_py["reglas"]) + list(reglas_go)))
    t = pd.DataFrame({"python": [rep_py["reglas"].get(c) for c in claves], "go": [reglas_go.get(c) for c in claves]}, index=claves)
    t["igual"] = t["python"] == t["go"]
    display(t)
    faltan = [c for c in claves if c not in reglas_go or c not in rep_py["reglas"]]
    chequeo("conteos por regla", bool(t["igual"].all()),
            "todos iguales" if t["igual"].all() else f"distintos en {list(t.index[~t['igual']])}; claves en un solo lado: {faltan}")
    fl_go = primera(lim_go, ["filas_largas"], {})
    for c in ["leidas", "procesadas", "descartadas"]:
        if c in fl_go:
            chequeo(f"filas largas {c}", fl_go[c] == rep_py["filas_largas"][c],
                    f"python {rep_py['filas_largas'][c]:,} · go {fl_go[c]:,}")
    fa_go = primera(lim_go, ["filas_anchas"], {})
    total_go = fa_go.get("total") if isinstance(fa_go, dict) else fa_go
    if total_go is not None:
        chequeo("filas anchas", total_go == len(ancho_py), f"python {len(ancho_py):,} · go {total_go:,}")
    marcas_go = primera(lim_go, ["marcas"], {})
    if marcas_go:
        print("Marcas (informativo): python", rep_py["marcas"], "· go", marcas_go)
else:
    chequeo("conteos por regla", False, "falta el bronze o reports/limpieza.json de Go", omitido=True)

# Silver celda por celda.
if ancho_py is not None and silver_go is not None:
    chequeo("cabecera del silver", list(silver_go.columns) == list(ancho_py.columns),
            f"go {list(silver_go.columns)}" + ("" if list(silver_go.columns) == list(ancho_py.columns) else f" · python {list(ancho_py.columns)}"))
    ordenado = silver_go[["estacion", "fecha_hora"]].equals(
        silver_go[["estacion", "fecha_hora"]].sort_values(["estacion", "fecha_hora"], kind="mergesort").reset_index(drop=True))
    chequeo("orden del silver (estación, fecha_hora)", ordenado, "ordenado" if ordenado else "Go no escribe en orden")
    m = ancho_py.merge(silver_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
    solo_py, solo_go = m[m["_merge"] == "left_only"], m[m["_merge"] == "right_only"]
    chequeo("claves del silver", len(solo_py) == 0 and len(solo_go) == 0,
            f"{(m['_merge'] == 'both').sum():,} en ambos · {len(solo_py):,} solo en Python · {len(solo_go):,} solo en Go")
    if len(solo_py) or len(solo_go):
        display(pd.concat([solo_py.head(5), solo_go.head(5)])[["estacion", "fecha_hora", "_merge"]])
    filas_dif = []
    ambos = m[m["_merge"] == "both"]
    for c in [c for c in ancho_py.columns[2:] if c in silver_go.columns]:
        a, b = ambos[c + "_py"].to_numpy(float), ambos[c + "_go"].to_numpy(float)
        nan_dif = np.isnan(a) != np.isnan(b)
        val_dif = ~np.isnan(a) & ~np.isnan(b) & (np.abs(a - b) > TOL_SILVER * np.maximum(1, np.abs(a)))
        malas = nan_dif | val_dif
        if malas.any():
            filas_dif.append(ambos.loc[malas, ["estacion", "fecha_hora", c + "_py", c + "_go"]].head(10))
        print(f"  {c}: {int(nan_dif.sum())} celdas con faltante distinto, {int(val_dif.sum())} con valor distinto")
    chequeo("valores del silver", not filas_dif, "idénticos" if not filas_dif else f"{len(filas_dif)} columnas con diferencias (abajo)")
    for d in filas_dif:
        display(d)
else:
    chequeo("silver celda por celda", False, "falta el bronze o el silver de Go", omitido=True)

## 3. Gold: receta sobre el silver de Python frente al gold de Go

In [ ]:
base_silver = ancho_py if ancho_py is not None else silver_go
gold_py, stats_py = al.construir_gold(base_silver, receta) if base_silver is not None else (None, None)
gold_go = al.leer_csv_contrato(GO["gold"]) if GO["gold"].exists() else None
if gold_py is not None and gold_go is not None:
    cab_ok = list(gold_go.columns) == ["estacion", "fecha_hora"] + FEATS
    chequeo("cabecera del gold", cab_ok, f"go {list(gold_go.columns)}")
    m = gold_py.merge(gold_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
    n_solo = int((m["_merge"] != "both").sum())
    chequeo("filas del gold (regla 8)", n_solo == 0,
            f"python {len(gold_py):,} · go {len(gold_go):,} · {n_solo} claves en un solo lado")
    ambos = m[m["_merge"] == "both"]
    D = np.abs(ambos[[f + "_py" for f in FEATS]].to_numpy(float) - ambos[[f + "_go" for f in FEATS]].to_numpy(float))
    peor = float(np.nanmax(D)) if D.size else float("nan")
    malas = (D > TOL_GOLD).any(axis=1)
    chequeo("valores del gold", not malas.any(), f"diferencia máxima {peor:.2e} (tolerancia {TOL_GOLD:g}); {int(malas.sum())} filas fuera"
            + (" (las filas no coinciden: con otras filas cambian la media y la desviación; resolver primero las claves)" if n_solo else ""))
    if malas.any():
        display(ambos.loc[malas].head(20))
    if silver_go is not None:
        # El paso silver -> gold aislado: la receta sobre el silver de Go frente al gold de Go.
        g2, _ = al.construir_gold(silver_go, receta)
        m2 = g2.merge(gold_go, on=["estacion", "fecha_hora"], how="outer", indicator=True, suffixes=("_py", "_go"))
        a2 = m2[m2["_merge"] == "both"]
        d2max = float(np.nanmax(np.abs(a2[[f + "_py" for f in FEATS]].to_numpy(float) - a2[[f + "_go" for f in FEATS]].to_numpy(float)))) if len(a2) else float("nan")
        chequeo("gold desde el silver de Go", int((m2["_merge"] != "both").sum()) == 0 and d2max <= TOL_GOLD,
                f"{int((m2['_merge'] != 'both').sum())} claves en un solo lado; diferencia máxima {d2max:.2e}")
    print(pd.Series(D.max(axis=0) if D.size else [], index=FEATS[: D.shape[1]] if D.size else None, name="máx |Δ| por feature"))
else:
    chequeo("gold", False, "falta el silver o el gold de Go", omitido=True)

if stats_py is not None and GO["estandarizacion"].exists():
    est_go = leer_json(GO["estandarizacion"])
    chequeo("receta en estandarizacion.json",
            est_go.get("features") == FEATS and (est_go.get("log1p") or []) == LOGS and est_go.get("nivel", NIVEL) == NIVEL,
            f"go features={est_go.get('features')} log1p={est_go.get('log1p')} nivel={est_go.get('nivel')}")
    chequeo("n de la estandarización", est_go.get("n") == stats_py["n"], f"python {stats_py['n']:,} · go {est_go.get('n')}")
    if NIVEL == "absoluto":
        mu_py, sd_py = np.array(stats_py["media"]), np.array(stats_py["desv"])
        mu_go, sd_go = np.array(est_go["media"], float), np.array(est_go["desv"], float)
    else:  # por estación: se aplanan en el mismo orden de estaciones
        ests = sorted(stats_py["media"])
        mu_py = np.concatenate([stats_py["media"][e] for e in ests]); sd_py = np.concatenate([stats_py["desv"][e] for e in ests])
        mu_go = np.concatenate([est_go["media"][e] for e in ests]); sd_go = np.concatenate([est_go["desv"][e] for e in ests])
    rel_mu = np.max(np.abs(mu_py - mu_go) / np.maximum(1, np.abs(mu_py)))
    rel_sd = np.max(np.abs(sd_py - sd_go) / np.maximum(1e-12, np.abs(sd_py)))
    chequeo("medias de la estandarización", rel_mu <= TOL_ESTAND, f"diferencia relativa máxima {rel_mu:.2e}")
    pista = ""
    n = stats_py["n"]
    if rel_sd > TOL_ESTAND and n > 1 and np.allclose(sd_go / sd_py, math.sqrt(n / (n - 1)), rtol=1e-6):
        pista = " (Go usa la desviación muestral n−1; el contrato es poblacional, n)"
    chequeo("desviaciones de la estandarización", rel_sd <= TOL_ESTAND, f"diferencia relativa máxima {rel_sd:.2e}{pista}")
else:
    chequeo("estandarización", False, "falta reports/estandarizacion.json de Go", omitido=True)

## 4. K-Means de Go

- **Asignaciones**: cada fila del gold de Go al centroide de Go más cercano (distancia euclídea al
  cuadrado, empate al menor índice: el contrato del TP) frente a `asignaciones.csv`.
- **Tamaños e inercia** recalculados.
- **Punto fijo**: si Lloyd convergió, cada centroide es la media de sus filas; además scikit-learn
  arrancando de los centroides de Go no debería moverlos.
- **scikit-learn independiente** (k-means++, 10 arranques, la semilla de la receta): mismos
  centroides tras emparejar etiquetas con la asignación húngara. Si difieren pero la inercia es casi
  la misma, es otro mínimo local, no un bug.

In [ ]:
km_go = leer_json(GO["kmeans"]) if GO["kmeans"].exists() else None
asig_go = al.leer_csv_contrato(GO["asignaciones"]) if GO["asignaciones"].exists() else None
gold_ref = gold_go if gold_go is not None else gold_py
Cz_go = None
if km_go is not None and gold_ref is not None:
    Cz_go = np.array(primera(km_go, ["centroides_estandarizados", "centroides", "centroides_z"]), dtype=float)
    X = gold_ref[FEATS].to_numpy(float)
    chequeo("forma de los centroides", Cz_go.shape == (K, len(FEATS)), f"go {Cz_go.shape} · receta ({K}, {len(FEATS)})")
    lab_py, d2 = al.asignar(X, Cz_go)
    if asig_go is not None:
        j = gold_ref[["estacion", "fecha_hora"]].assign(_lab=lab_py).merge(asig_go, on=["estacion", "fecha_hora"], how="outer", indicator=True)
        n_solo = int((j["_merge"] != "both").sum())
        dif = j[(j["_merge"] == "both") & (j["_lab"] != j["regimen"])]
        chequeo("asignaciones", n_solo == 0 and len(dif) == 0,
                f"{len(dif):,} filas con otro régimen · {n_solo} claves en un solo lado de {len(asig_go):,}")
        if len(dif):
            display(dif.head(20))
        lab_go = j.loc[j["_merge"] == "both", "regimen"].to_numpy(int) if n_solo == 0 else lab_py
    else:
        chequeo("asignaciones", False, "falta data/gold/asignaciones.csv", omitido=True)
        lab_go = lab_py
    tam_go = primera(km_go, ["tamanos", "tamaños", "conteos"])
    if tam_go is not None:
        chequeo("tamaños", list(np.bincount(lab_py, minlength=K)) == list(tam_go),
                f"python {np.bincount(lab_py, minlength=K).tolist()} · go {list(tam_go)}")
    if "inercia" in km_go:
        rel = abs(d2.sum() - km_go["inercia"]) / max(1e-12, abs(km_go["inercia"]))
        chequeo("inercia", rel <= 1e-9, f"python {d2.sum():.6f} · go {km_go['inercia']:.6f} (rel {rel:.1e})")
    medias = np.array([X[lab_go == j].mean(axis=0) if (lab_go == j).any() else Cz_go[j] for j in range(K)])
    pf = float(np.abs(medias - Cz_go).max())
    chequeo("punto fijo de Lloyd", pf <= TOL_PUNTO_FIJO, f"máx |media de las filas − centroide| = {pf:.2e}")
    km_fijo = KMeans(n_clusters=K, init=Cz_go, n_init=1, max_iter=300, tol=0).fit(X)
    mov = float(np.abs(km_fijo.cluster_centers_ - Cz_go).max())
    chequeo("scikit-learn desde los centroides de Go", mov <= TOL_PUNTO_FIJO,
            f"se mueven {mov:.2e} en {km_fijo.n_iter_} iteraciones")
    km_sk = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=int(receta.get("semilla", 42))).fit(X)
    perm = al.emparejar(Cz_go, km_sk.cluster_centers_)
    dmax = float(np.abs(km_sk.cluster_centers_[perm] - Cz_go).max())
    ari = adjusted_rand_score(lab_go, km_sk.labels_)
    rel_in = (km_sk.inertia_ - d2.sum()) / d2.sum()
    det = f"máx |Δ| tras emparejar {dmax:.2e} (tolerancia {TOL_SKLEARN:g}); ARI {ari:.4f}; inercia sklearn/go − 1 = {rel_in:+.2%}"
    aviso = False
    if dmax > TOL_SKLEARN and pf <= TOL_PUNTO_FIJO:
        aviso = True  # Lloyd de Go está bien (punto fijo): la diferencia es de inicialización
        if abs(rel_in) < 1e-3:
            det += ". Otro mínimo local de igual calidad: no es un bug"
        elif rel_in < 0:
            det += (". scikit-learn (10 arranques) encuentra una partición de menor inercia: Go quedó en un mínimo "
                    "local peor con su único k-means++. Lloyd de Go es correcto (punto fijo OK); es la inicialización. "
                    "Mirar la estabilidad por semillas del notebook 02 o probar otra semilla")
        else:
            det += ". Go encuentra una partición de menor inercia que scikit-learn"
    chequeo("centroides frente a scikit-learn", dmax <= TOL_SKLEARN, det, aviso=aviso)
    print("Emparejamiento go → sklearn:", perm.tolist())
    ref02 = REPORTS / "centroides_sklearn.json"
    if ref02.exists():
        r02 = leer_json(ref02)
        if r02["receta"]["features"] == FEATS and r02["receta"]["k"] == K and r02["receta"]["nivel"] == NIVEL:
            C02 = np.array(r02["centroides_estandarizados"])
            p2 = al.emparejar(Cz_go, C02)
            print(f"Referencia del notebook 02 ({r02['fuente_silver']}): máx |Δ| {np.abs(C02[p2] - Cz_go).max():.2e} "
                  "(informativo: puede venir del silver prototipo)")
    Co_go = primera(km_go, ["centroides_desestandarizados", "centroides_originales"])
    if Co_go is not None and NIVEL == "absoluto" and GO["estandarizacion"].exists():
        Co_py = al.desestandarizar(Cz_go, leer_json(GO["estandarizacion"]))
        rel = float(np.max(np.abs(Co_py - np.array(Co_go)) / np.maximum(1, np.abs(Co_py))))
        chequeo("centroides des-estandarizados", rel <= 1e-9, f"diferencia relativa máxima {rel:.2e}")
else:
    chequeo("K-Means de Go", False, "falta reports/kmeans.json de Go", omitido=True)

## 5. `regimenes.json`

In [ ]:
silver_fig = silver_go if silver_go is not None else ancho_py
if GO["regimenes"].exists() and asig_go is not None and silver_fig is not None:
    rg = leer_json(GO["regimenes"])
    lista = rg["regimenes"] if isinstance(rg, dict) else rg
    j = asig_go.merge(silver_fig, on=["estacion", "fecha_hora"], how="left")
    params = [c for c in silver_fig.columns if c not in ("estacion", "fecha_hora")]
    perfil_py = j.groupby("regimen")[params].mean()
    hora_py = pd.crosstab(j["regimen"], j["fecha_hora"].dt.hour).reindex(columns=range(24), fill_value=0)
    est_py = pd.crosstab(j["regimen"], j["estacion"])
    malos = []
    for r in lista:
        rid = int(primera(r, ["regimen", "id"]))
        if "n" in r and r["n"] != int((asig_go["regimen"] == rid).sum()):
            malos.append(f"R{rid} n")
        perfil = primera(r, ["perfil", "medias"], {})
        for p, v in perfil.items():
            ref = perfil_py.loc[rid, p] if p in perfil_py else np.nan
            if (v is None) != bool(np.isnan(ref)) or (v is not None and abs(v - ref) > 1e-9 * max(1, abs(ref))):
                malos.append(f"R{rid} perfil {p}")
        exc_py = bool((perfil_py.loc[rid].get("pm2_5", 0) or 0) > al.ECA["pm2_5"] or (perfil_py.loc[rid].get("pm10", 0) or 0) > al.ECA["pm10"])
        if "excede_eca" in r and bool(r["excede_eca"]) != exc_py:
            malos.append(f"R{rid} etiqueta ECA")
        if "por_hora" in r and list(r["por_hora"]) != hora_py.loc[rid].tolist():
            malos.append(f"R{rid} por hora")
        if "por_estacion" in r and {k: int(v) for k, v in r["por_estacion"].items() if v} != {k: int(v) for k, v in est_py.loc[rid].items() if v}:
            malos.append(f"R{rid} por estación")
    chequeo("regimenes.json", not malos, "perfil, etiqueta ECA y distribuciones iguales" if not malos else f"difieren: {malos}")
else:
    chequeo("regimenes.json", False, "falta regimenes.json, asignaciones o silver", omitido=True)

## 6. Resumen de la auditoría

In [ ]:
resumen = pd.DataFrame(CHEQUEOS)
display(resumen)
n_dif = int((resumen["estado"] == "DIFERENCIA").sum())
hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
L = ["# Auditoría de las salidas de Go", "",
     f"> Generado por `notebooks/03_auditoria_figuras.ipynb` el {hoy}. Receta: `{origen_receta}`. "
     f"Limpieza: {origen_cfg}. Tolerancias: silver {TOL_SILVER:g} (relativa), gold {TOL_GOLD:g}, "
     f"estandarización {TOL_ESTAND:g}, punto fijo {TOL_PUNTO_FIJO:g}, scikit-learn {TOL_SKLEARN:g}."]
if SINTETICO:
    L += ["", "> **DATOS SINTÉTICOS**: las «salidas de Go» las escribió `notebooks/sintetico.py`."]
L += ["", f"**{(resumen['estado'] == 'OK').sum()} OK, {n_dif} con diferencias, {(resumen['estado'] == 'AVISO').sum()} avisos, "
      f"{(resumen['estado'] == 'OMITIDO').sum()} omitidas.** DIFERENCIA = bug en Go o en Python; AVISO = no es un bug pero hay "
      "que mirarlo (p. ej. otro mínimo local).", "",
      al.tabla_md(resumen)]
(REPORTS / "auditoria.md").write_text("\n".join(L) + "\n", encoding="utf-8")
(REPORTS / "auditoria.json").write_text(json.dumps({"fecha": hoy, "receta": receta, "chequeos": CHEQUEOS}, indent=2,
                                                   ensure_ascii=False, default=str), encoding="utf-8")
print("Escrito", REPORTS / "auditoria.md")
if n_dif:
    print(f"AVISO: {n_dif} comprobaciones con diferencias: hay un bug en Go o en Python.")

## 7. Figuras del informe

Sobre las salidas de Go (silver, asignaciones y centroides). Si Go todavía no corrió, los regímenes
salen de scikit-learn sobre el gold de Python y cada figura lo dice. Colores fijos por régimen (el
régimen 3 es siempre del mismo color); el ECA en rojo con su etiqueta.

In [ ]:
if asig_go is not None and Cz_go is not None:
    FUENTE_REG = "Go"
    asig, Cz = asig_go, Cz_go
    stats_fig = leer_json(GO["estandarizacion"]) if GO["estandarizacion"].exists() else stats_py
else:
    FUENTE_REG = "scikit-learn (Go todavía no corrió)"
    km_tmp = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=int(receta.get("semilla", 42))).fit(gold_py[FEATS].to_numpy(float))
    asig = gold_py[["estacion", "fecha_hora"]].assign(regimen=km_tmp.labels_)
    Cz, stats_fig = km_tmp.cluster_centers_, stats_py
print("Regímenes de:", FUENTE_REG)

J = asig.merge(silver_fig, on=["estacion", "fecha_hora"], how="left")
perfil = J.groupby("regimen")[[c for c in silver_fig.columns if c not in ("estacion", "fecha_hora")]].mean().reindex(range(K))
perfil["fraccion_fina"] = (J["pm2_5"] / J["pm10"].where(J["pm10"] > 0)).clip(0, 1).groupby(J["regimen"]).mean() if {"pm2_5", "pm10"} <= set(J) else np.nan
tam = J["regimen"].value_counts().reindex(range(K), fill_value=0)
excede = ((perfil.get("pm2_5", pd.Series(0, index=perfil.index)).fillna(0) > al.ECA["pm2_5"]) |
          (perfil.get("pm10", pd.Series(0, index=perfil.index)).fillna(0) > al.ECA["pm10"]))
orden_r = list(perfil["pm10"].fillna(-1).sort_values(ascending=False).index) if "pm10" in perfil else list(range(K))
ETIQ = {r: f"R{r}" + (" · excede ECA" if excede[r] else "") for r in range(K)}
NOTA_FUENTE = f"Regímenes: {FUENTE_REG}. Receta: {', '.join(FEATS)}; log1p en {', '.join(LOGS) or 'ninguna'}; nivel {NIVEL}; k = {K}."
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]
display(perfil.assign(horas=tam, excede=excede).round(2))

In [ ]:
# Figura 1. Perfiles por régimen: centroides (z) con su valor en unidades originales, y PM frente al ECA.
Co = al.desestandarizar(Cz, stats_fig) if NIVEL == "absoluto" else None
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 0.55 * K + 2.4), gridspec_kw={"width_ratios": [1.25, 1]})
Z = Cz[orden_r]
lim = max(1.0, min(3.0, float(np.abs(Z).max())))
im = a1.imshow(Z, cmap="aire_div", vmin=-lim, vmax=lim, aspect="auto")
a1.set_xticks(range(len(FEATS)), [f"{al.NOMBRE_TXT.get(f, f)}\n({al.UNIDAD_TXT.get(f, '')})" for f in FEATS])
a1.set_yticks(range(K), [ETIQ[r] for r in orden_r])
a1.grid(False)
for i, r in enumerate(orden_r):
    for jf, f in enumerate(FEATS):
        val = Co[r, jf] if Co is not None else perfil.loc[r, f] if f in perfil else np.nan
        txt = f"{val:.2f}" if f == "fraccion_fina" else f"{val:.0f}" if abs(val) >= 10 else f"{val:.1f}"
        a1.text(jf, i, txt, ha="center", va="center", fontsize=9, color="white" if abs(Z[i, jf]) > 0.6 * lim else al.TINTA)
fig.colorbar(im, ax=a1, shrink=0.8, label="centroide estandarizado (z)")
a1.set_title("Centroide de cada régimen", loc="left", pad=26)
y = np.arange(K)
for jp, (p, color) in enumerate([("pm10", "#25313C"), ("pm2_5", "#8c8b86")]):
    if p in perfil:
        a2.barh(y + (0.2 if jp == 0 else -0.2), perfil.loc[orden_r, p], height=0.38, color=color, label=f"{al.NOMBRE_TXT[p]} medio")
        a2.axvline(al.ECA[p], color=al.CRITICO, linewidth=1.1, linestyle="--")
        a2.text(al.ECA[p], -0.62, f"ECA {al.NOMBRE_TXT[p]}\n24 h = {al.ECA[p]:.0f}", color=al.CRITICO, fontsize=7.5,
                ha="center", va="bottom", clip_on=False)
a2.set_yticks(y, [f"{ETIQ[r]} · {100 * tam[r] / tam.sum():.1f} % de las horas" for r in orden_r], fontsize=8)
a2.invert_yaxis()
a2.set_ylim(K - 0.4, -0.6)
a2.set_xlabel("µg/m³ (media de las horas del régimen)")
a2.set_title("Material particulado frente al ECA", loc="left", pad=26)
a2.grid(axis="y", visible=False)
a2.legend(loc="lower right")
fig.suptitle("Perfil de los regímenes de calidad del aire", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
al.nota_al_pie(fig, NOTA_FUENTE + " Valores de la izquierda: centroide en unidades originales.")
guardar(fig, "fig_perfiles_regimenes.png")

In [ ]:
# Figura 2. Régimen × hora del día: % de las horas de cada régimen que caen en cada hora.
H = pd.crosstab(J["regimen"], J["fecha_hora"].dt.hour).reindex(index=orden_r, columns=range(24), fill_value=0)
Hp = 100 * H.div(H.sum(axis=1).replace(0, np.nan), axis=0)
fig, ax = plt.subplots(figsize=(11, 0.45 * K + 1.8))
im = ax.imshow(Hp.to_numpy(), cmap="aire_seq", aspect="auto", vmin=0)
ax.set_xticks(range(24), [f"{h:02d}" for h in range(24)], fontsize=8)
ax.set_yticks(range(K), [ETIQ[r] for r in orden_r])
ax.set_xlabel("Hora del día (hora local de Lima)")
ax.grid(False)
fig.colorbar(im, ax=ax, label="% de las horas del régimen", shrink=0.85)
ax.set_title("¿A qué hora aparece cada régimen?", loc="left")
al.nota_al_pie(fig, NOTA_FUENTE + " Cada fila suma 100 %; una fila plana es un régimen sin hora preferida.")
guardar(fig, "fig_regimen_hora.png")

# Figura 3. Estación × régimen: % de las horas de cada estación en cada régimen.
E = pd.crosstab(J["estacion"], J["regimen"]).reindex(columns=orden_r, fill_value=0)
Ep = 100 * E.div(E.sum(axis=1), axis=0)
dep = {}
if largo is not None and "departamento" in largo:
    dd = largo.groupby(["estacion", "departamento"], observed=True).size().reset_index().sort_values(0, ascending=False)
    dep = {str(e).strip(): al.NOMBRE_REGION.get(al.region(d), str(d).title()) for e, d in zip(dd["estacion"], dd["departamento"])}
    dep = {e: dep[e] for e in dep}
Ep = Ep.loc[sorted(Ep.index, key=lambda e: (dep.get(e, ""), e))]
fig, ax = plt.subplots(figsize=(1.0 * K + 4.5, 0.32 * len(Ep) + 1.8))
im = ax.imshow(Ep.to_numpy(), cmap="aire_seq", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(K), [ETIQ[r].replace(" · ", "\n") for r in orden_r], fontsize=8)
ax.set_yticks(range(len(Ep)), [f"{e} ({dep[e]})" if e in dep else e for e in Ep.index], fontsize=8)
ax.grid(False)
if len(Ep) <= 40:
    for i in range(len(Ep)):
        for j in range(K):
            v = Ep.iat[i, j]
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 55 else al.TINTA)
fig.colorbar(im, ax=ax, label="% de las horas de la estación", shrink=0.85)
ax.set_title("¿Qué régimen domina en cada estación?", loc="left")
al.nota_al_pie(fig, NOTA_FUENTE + " Cada fila suma 100 %.")
guardar(fig, "fig_regimen_estacion.png")

In [ ]:
# Figura 4 (la estrella). Un episodio real: PM2.5 y PM10 de una estación durante unos días, con el
# fondo coloreado por régimen y las líneas del ECA.
EPISODIO_DIAS = 7
ESTACION_EPISODIO = None   # None: la ventana con más horas de promedio de 24 h sobre el ECA

rej = al.regularizar(silver_fig, [c for c in ["pm2_5", "pm10"] if c in silver_fig])
mv = al.movil_24h(rej)
rej = rej.merge(mv[["estacion", "fecha_hora", "excede_24h"]], on=["estacion", "fecha_hora"]).merge(
    asig, on=["estacion", "fecha_hora"], how="left")
rej["dia"] = rej["fecha_hora"].dt.normalize()
diario = rej.groupby(["estacion", "dia"]).agg(exc=("excede_24h", "sum"), asignadas=("regimen", "count"), horas=("fecha_hora", "size"))
mejor = None
for est, g in diario.groupby(level=0):
    if ESTACION_EPISODIO and est != ESTACION_EPISODIO:
        continue
    g = g.droplevel(0).asfreq("D", fill_value=0)
    puntaje = g["exc"].rolling(EPISODIO_DIAS).sum()
    cobertura = g["asignadas"].rolling(EPISODIO_DIAS).sum() / (24 * EPISODIO_DIAS)
    puntaje = puntaje.where(cobertura >= 0.6)
    if puntaje.notna().any() and (mejor is None or puntaje.max() > mejor[2]):
        mejor = (est, puntaje.idxmax(), float(puntaje.max()))
if mejor is None:
    print("No hay una ventana con regímenes asignados suficientes para la figura del episodio.")
else:
    est, fin_dia, punt = mejor
    ini = fin_dia - pd.Timedelta(days=EPISODIO_DIAS - 1)
    w = rej[(rej["estacion"] == est) & (rej["dia"] >= ini) & (rej["dia"] <= fin_dia)].reset_index(drop=True)
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True, gridspec_kw={"height_ratios": [6, 0.55], "hspace": 0.08})
    t = w["fecha_hora"]
    paso = pd.Timedelta(hours=1)
    reg = w["regimen"]
    corte = (reg != reg.shift()).cumsum()
    for _, tramo in w.groupby(corte):
        r = tramo["regimen"].iloc[0]
        if pd.isna(r):
            continue
        x0, x1 = tramo["fecha_hora"].iloc[0], tramo["fecha_hora"].iloc[-1] + paso
        ax.axvspan(x0, x1, color=al.color_regimen(int(r)), alpha=0.16, linewidth=0)
        axr.axvspan(x0, x1, color=al.color_regimen(int(r)), alpha=0.9, linewidth=0)
    for p, color, lw in [("pm10", "#0b0b0b", 1.6), ("pm2_5", "#52514e", 1.2)]:
        if p in w:
            ax.plot(t, w[p], color=color, linewidth=lw, label=f"{al.NOMBRE_TXT[p]} horario")
            ultimo = w[p].last_valid_index()
            if ultimo is not None:
                ax.annotate(al.NOMBRE_TXT[p], (t[ultimo], w[p][ultimo]), xytext=(5, 0), textcoords="offset points",
                            fontsize=9, color=color, va="center")
            ax.axhline(al.ECA[p], color=al.CRITICO, linewidth=1.1, linestyle="--")
            ax.text(t.iloc[0], al.ECA[p], f" ECA {al.NOMBRE_TXT[p]} (24 h) = {al.ECA[p]:.0f} µg/m³", color=al.CRITICO,
                    fontsize=8, va="bottom")
    ax.set_ylabel("µg/m³")
    ax.set_ylim(bottom=0)
    ax.grid(axis="x", visible=False)
    presentes = [int(r) for r in pd.unique(reg.dropna())]
    orden_leyenda = [r for r in orden_r if r in presentes]
    ax.legend(handles=[Patch(color=al.color_regimen(r), alpha=0.6, label=ETIQ[r]) for r in orden_leyenda],
              loc="upper left", bbox_to_anchor=(1.01, 1), title="Régimen (fondo)", fontsize=8)
    axr.set_yticks([])
    axr.set_ylabel("régimen", rotation=0, ha="right", va="center", fontsize=8)
    axr.grid(False)
    axr.xaxis.set_major_locator(mdates.DayLocator())
    axr.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: (lambda d: f"{d.day} {MESES[d.month - 1]}")(mdates.num2date(x))))
    ax.set_title(f"Un episodio en {est}: {ini.day} {MESES[ini.month - 1]} – {fin_dia.day} {MESES[fin_dia.month - 1]} {fin_dia.year}",
                 loc="left", fontsize=13)
    al.nota_al_pie(fig, NOTA_FUENTE + "\nEl ECA es un promedio de 24 h: una hora sobre la línea no es por sí sola un "
                   "incumplimiento. Sin fondo: horas sin régimen (falta alguna feature).")
    guardar(fig, "fig_episodio.png")
    print(f"Episodio: {est}, {ini:%Y-%m-%d} a {fin_dia:%Y-%m-%d}, {punt:.0f} horas con el promedio de 24 h sobre el ECA")

In [ ]:
# Figura 5. Cobertura: estaciones y horas válidas por región (las 25 regiones, también las que no tienen).
if largo is not None and "departamento" in largo:
    d_est = (largo.groupby(["estacion", "departamento"], observed=True).size().reset_index()
             .sort_values(0, ascending=False).drop_duplicates("estacion"))
    d_est["estacion"] = d_est["estacion"].astype(str).str.strip()
    d_est["region"] = [al.region(d) for d in d_est["departamento"]]
    validas = set(silver_fig["estacion"])
    d_est = d_est[d_est["estacion"].isin(validas)]
    hpm = silver_fig[[c for c in ["pm2_5", "pm10"] if c in silver_fig]].notna().any(axis=1).groupby(silver_fig["estacion"]).sum()
    d_est["horas_pm"] = d_est["estacion"].map(hpm).fillna(0)
    por_reg = d_est.groupby("region").agg(estaciones=("estacion", "size"), horas_pm=("horas_pm", "sum"))
    regiones = sorted(set(al.REGIONES_PERU) | set(por_reg.index))
    por_reg = por_reg.reindex(regiones, fill_value=0)
    por_reg = por_reg.assign(_n=por_reg.index).sort_values(
        ["estaciones", "horas_pm", "_n"], ascending=[False, False, True]).drop(columns="_n")
    con = int((por_reg["estaciones"] > 0).sum())
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 0.22 * len(por_reg) + 1.6), sharey=True, gridspec_kw={"wspace": 0.08})
    y = np.arange(len(por_reg))
    nombres = [al.NOMBRE_REGION.get(r, r.title()) for r in por_reg.index]
    colores = [al.PALETA[0] if n > 0 else "#d9d8d3" for n in por_reg["estaciones"]]
    a1.barh(y, por_reg["estaciones"], color=colores, height=0.65)
    for yi, n in zip(y, por_reg["estaciones"]):
        a1.text(n + 0.05, yi, f"{int(n)}", va="center", fontsize=8, color=al.TINTA if n else "#8c8b86")
    a1.set_yticks(y, nombres, fontsize=8)
    for lab, n in zip(a1.get_yticklabels(), por_reg["estaciones"]):
        lab.set_color(al.TINTA if n else "#8c8b86")
    a1.set_xlabel("estaciones con datos válidos")
    a1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    a1.invert_yaxis()
    a1.set_title("Estaciones por región", loc="left")
    a2.barh(y, por_reg["horas_pm"] / 1000, color=colores, height=0.65)
    a2.set_xlabel("miles de horas con PM2.5 o PM10 válido")
    a2.set_title("Horas con dato de material particulado", loc="left")
    for a in (a1, a2):
        a.grid(axis="y", visible=False)
    fig.suptitle(f"Cobertura de OEFA: {int(por_reg['estaciones'].sum())} estaciones en {con} de las {len(regiones)} regiones",
                 x=0.01, ha="left", fontweight="bold", fontsize=13)
    al.nota_al_pie(fig, "Una región sin estación no es una región con aire limpio: es una región sin medición.", 8)
    guardar(fig, "fig_cobertura_departamento.png")
else:
    print("Sin bronze (o sin columna de departamento): no se dibuja la cobertura.")

In [ ]:
# Figura 6. Curva del codo y silueta sobre el gold (submuestra sistemática si es grande), k = 2..10.
Xc = gold_ref[FEATS].to_numpy(float)
paso_c = max(1, math.ceil(len(Xc) / 200_000))
Xc = Xc[::paso_c]
ks, inercias, siluetas = list(range(2, 11)), [], []
for k in ks:
    m = KMeans(n_clusters=k, init="k-means++", n_init=3, random_state=int(receta.get("semilla", 42))).fit(Xc)
    inercias.append(m.inertia_ * paso_c)
    siluetas.append(silhouette_score(Xc, m.labels_, sample_size=min(10_000, len(Xc)), random_state=0))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(ks, inercias, marker="o", color=al.PALETA[0])
a1.set_title("Inercia (suma de distancias al cuadrado)", loc="left")
a1.set_ylabel("inercia")
a2.plot(ks, siluetas, marker="o", color=al.PALETA[0])
a2.set_title("Silueta media (10 000 puntos)", loc="left")
for a in (a1, a2):
    a.axvline(K, color=al.CRITICO, linewidth=1, linestyle="--")
    a.set_xlabel("k (número de regímenes)")
    a.set_xticks(ks)
a1.annotate(f"k = {K} (receta)", (K, inercias[ks.index(K)] if K in ks else inercias[0]), xytext=(8, 8),
            textcoords="offset points", color=al.CRITICO, fontsize=9)
fig.suptitle("Curva del codo del K-Means", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
al.nota_al_pie(fig, f"scikit-learn sobre el gold ({al.miles(len(Xc))} filas{'' if paso_c == 1 else f', cada {paso_c}.ª'}); "
               f"k = {K} es el de configs/kmeans.json. Go reporta solo la inercia; la silueta en Go llega en la PC4.")
guardar(fig, "fig_codo.png")
print("Figuras:", sorted(p.name for p in FIGURAS.glob("*.png")))